# Notebook 05: Prediction Model – High-Risk Parking Event Classifier
**Project:** Parking-Induced Congestion AI System

This notebook:
- Merges cluster-level risk scores into record-level parking violations
- Creates a binary target: **High Risk (1)** = High or Critical category, **Low Risk (0)** otherwise
- Trains a **Random Forest Classifier**
- Evaluates with accuracy, precision, recall, F1, confusion matrix
- Plots feature importance
- Saves `ml_predictions.csv`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries imported.')

Libraries imported.


## 2. Define Paths

In [2]:
CLUSTERED_CSV    = '../data/processed/parking_violations_with_clusters.csv'
RISK_CSV         = '../data/processed/congestion_risk_scores.csv'
PREDICTIONS_CSV  = '../data/processed/ml_predictions.csv'
CHARTS_DIR       = '../data/processed/charts/'

os.makedirs(CHARTS_DIR, exist_ok=True)
print('Paths ready.')

Paths ready.


## 3. Load Data

In [3]:
df_records = pd.read_csv(CLUSTERED_CSV, low_memory=False)
df_risk    = pd.read_csv(RISK_CSV, low_memory=False)

if 'datetime' in df_records.columns:
    df_records['datetime'] = pd.to_datetime(df_records['datetime'], errors='coerce')

print(f'Records shape : {df_records.shape}')
print(f'Risk scores   : {df_risk.shape}')
display(df_records.head())

Records shape : (298450, 32)
Risk scores   : (349, 19)


,id,latitude,longitude,location_or_junction,vehicle_type,vehicle_type.1,description,violation_type,offence_code,created_datetime,...,validation_status,validation_timestamp,datetime,hour,day,month,day_of_week,weekday_name,is_weekend,cluster_id
0,FKID000000,12.925557,77.618665,"18Th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""Wrong Parking"",""Parking Near Road Crossing""]","[112,104]",2023-11-20 00:28:46+00,...,approved,2023-11-30 03:08:24.818+00,2023-11-20 00:28:46+00:00,0.0,20.0,11.0,0.0,Monday,0,0
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""No Parking""]",[113],2023-11-24 22:46:46+00,...,NaN,NaN,2023-11-24 22:46:46+00:00,22.0,24.0,11.0,4.0,Friday,0,1
2,FKID000002,12.925449,77.618504,"Koramangala 2Nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""Wrong Parking"",""Parking In A Main Road""]","[112,107]",2023-11-20 00:27:46+00,...,approved,2023-11-30 03:08:56.998+00,2023-11-20 00:27:46+00:00,0.0,20.0,11.0,0.0,Monday,0,0
3,FKID000003,12.956521,77.518618,"6Th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""No Parking""]",[113],2023-11-16 06:47:46+00,...,approved,2023-11-18 23:35:12.428+00,2023-11-16 06:47:46+00:00,6.0,16.0,11.0,3.0,Thursday,0,2
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""No Parking""]",[113],2023-11-22 04:56:46+00,...,approved,2023-11-30 03:11:32.796+00,2023-11-22 04:56:46+00:00,4.0,22.0,11.0,2.0,Wednesday,0,3


## 4. Merge Risk Scores into Records

In [4]:
risk_cols = ['cluster_id', 'risk_score', 'risk_category']
risk_cols = [c for c in risk_cols if c in df_risk.columns]
df_risk_subset = df_risk[risk_cols].copy()

df = df_records.merge(df_risk_subset, on='cluster_id', how='left')

# Records with cluster_id = -1 (noise) won't have risk info; fill with Low
df['risk_category'] = df['risk_category'].fillna('Low')
df['risk_score']    = df['risk_score'].fillna(0)

print(f'Merged dataset shape: {df.shape}')
print('Risk category distribution in merged data:')
print(df['risk_category'].value_counts())

Merged dataset shape: (298450, 34)
Risk category distribution in merged data:
risk_category
High      186628
Low        57028
Medium     54794
Name: count, dtype: int64


## 5. Create Binary Target

In [5]:
df['high_risk'] = df['risk_category'].isin(['High', 'Critical']).astype(int)

class_counts = df['high_risk'].value_counts()
print('Target class distribution:')
print(class_counts)
print(f'  High-risk (1) : {class_counts.get(1, 0):,} ({class_counts.get(1,0)/len(df)*100:.1f}%)')
print(f'  Low-risk  (0) : {class_counts.get(0, 0):,} ({class_counts.get(0,0)/len(df)*100:.1f}%)')

if class_counts.get(1, 0) < 10:
    print('\nWARNING: Very few high-risk examples (<10). Model may not generalize well.')
if len(class_counts) < 2:
    print('\nCRITICAL: Only one class present in target. Model training is not possible.')
    print('Saving records with a dummy prediction column and exiting gracefully.')
    df['actual_high_risk']              = df['high_risk']
    df['predicted_high_risk']           = df['high_risk']
    df['predicted_probability_high_risk'] = df['high_risk'].astype(float)
    pred_cols = ['latitude','longitude','datetime','violation_type','police_station',
                 'location_or_junction','cluster_id','actual_high_risk',
                 'predicted_high_risk','predicted_probability_high_risk']
    pred_cols = [c for c in pred_cols if c in df.columns]
    df[pred_cols].to_csv(PREDICTIONS_CSV, index=False)
    print(f'Saved to {PREDICTIONS_CSV} (no model trained — single class).')
    raise SystemExit(0)

Target class distribution:
high_risk
1    186628
0    111822
Name: count, dtype: int64
  High-risk (1) : 186,628 (62.5%)
  Low-risk  (0) : 111,822 (37.5%)


## 6. Feature Engineering

In [6]:
feature_cols = []

# Numeric features
for col in ['hour', 'day_of_week', 'month', 'latitude', 'longitude']:
    if col in df.columns and df[col].notna().sum() > 0:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())
        feature_cols.append(col)
        print(f'  Numeric feature added: {col}')

# Categorical features – label encode
le_dict = {}
for cat_col in ['police_station', 'violation_type', 'location_or_junction', 'vehicle_type']:
    if cat_col in df.columns and df[cat_col].notna().sum() > 0:
        enc_col = f'{cat_col}_enc'
        le = LabelEncoder()
        df[enc_col] = le.fit_transform(df[cat_col].astype(str).fillna('Unknown'))
        le_dict[cat_col] = le
        feature_cols.append(enc_col)
        print(f'  Encoded feature added: {enc_col} ({df[cat_col].nunique()} classes)')

print(f'\nTotal features: {len(feature_cols)}')
print('Features:', feature_cols)

  Numeric feature added: hour
  Numeric feature added: day_of_week
  Numeric feature added: month
  Numeric feature added: latitude
  Numeric feature added: longitude
  Encoded feature added: police_station_enc (54 classes)
  Encoded feature added: violation_type_enc (991 classes)
  Encoded feature added: location_or_junction_enc (10941 classes)
  Encoded feature added: vehicle_type_enc (231890 classes)

Total features: 9
Features: ['hour', 'day_of_week', 'month', 'latitude', 'longitude', 'police_station_enc', 'violation_type_enc', 'location_or_junction_enc', 'vehicle_type_enc']


## 7. Prepare X and y

In [7]:
if len(feature_cols) == 0:
    raise ValueError('No features available for model training. Check upstream notebooks.')

X = df[feature_cols].copy()
y = df['high_risk'].copy()

# Fill any remaining NaN
X = X.fillna(X.median(numeric_only=True))

print(f'X shape: {X.shape}')
print(f'y distribution:\n{y.value_counts()}')

X shape: (298450, 9)
y distribution:
high_risk
1    186628
0    111822
Name: count, dtype: int64


## 8. Train/Test Split

In [8]:
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print('Stratified split successful.')
except ValueError as e:
    print(f'Stratified split failed ({e}). Using non-stratified split.')
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train class distribution:\n{y_train.value_counts()}')
print(f'Test class distribution:\n{y_test.value_counts()}')

Stratified split successful.
Train: (238760, 9), Test: (59690, 9)
Train class distribution:
high_risk
1    149302
0     89458
Name: count, dtype: int64
Test class distribution:
high_risk
1    37326
0    22364
Name: count, dtype: int64


## 9. Train Random Forest Classifier

In [9]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'   # handle class imbalance
)

print('Training Random Forest...')
rf.fit(X_train, y_train)
print('Training complete.')

Training Random Forest...
Training complete.


## 10. Evaluate Model

In [10]:
y_pred      = rf.predict(X_test)
y_prob      = rf.predict_proba(X_test)[:, 1]

acc   = accuracy_score(y_test, y_pred)
prec  = precision_score(y_test, y_pred, zero_division=0)
rec   = recall_score(y_test, y_pred, zero_division=0)
f1    = f1_score(y_test, y_pred, zero_division=0)

print('=' * 50)
print('MODEL EVALUATION')
print('=' * 50)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk'], zero_division=0))

MODEL EVALUATION
  Accuracy  : 0.9969
  Precision : 0.9955
  Recall    : 0.9996
  F1 Score  : 0.9976

Classification Report:
              precision    recall  f1-score   support

    Low Risk       1.00      0.99      1.00     22364
   High Risk       1.00      1.00      1.00     37326

    accuracy                           1.00     59690
   macro avg       1.00      1.00      1.00     59690
weighted avg       1.00      1.00      1.00     59690



## 11. Confusion Matrix

In [11]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Low Risk','High Risk'],
            yticklabels=['Low Risk','High Risk'])
ax.set_title('Confusion Matrix – High Risk Parking Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
cm_path = os.path.join(CHARTS_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Confusion matrix saved: {cm_path}')

Confusion matrix saved: ../data/processed/charts/confusion_matrix.png


## 12. Feature Importance

In [12]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, palette='viridis', ax=ax)
ax.set_title('Feature Importance – Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')
plt.tight_layout()
fi_path = os.path.join(CHARTS_DIR, 'feature_importance.png')
plt.savefig(fi_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Feature importance chart saved: {fi_path}')

print('\nTop Features:')
display(importances.reset_index().rename(columns={'index':'feature', 0:'importance'}))

Feature importance chart saved: ../data/processed/charts/feature_importance.png

Top Features:


,feature,importance
0,longitude,0.461279
1,latitude,0.296157
2,police_station_enc,0.199359
3,location_or_junction_enc,0.030990
4,violation_type_enc,0.007109
5,hour,0.003231
6,month,0.000806
7,vehicle_type_enc,0.000686
8,day_of_week,0.000383


## 13. Generate Predictions on Full Dataset

In [13]:
df['predicted_high_risk']            = rf.predict(X)
df['predicted_probability_high_risk'] = rf.predict_proba(X)[:, 1].round(4)
df['actual_high_risk']               = df['high_risk']

pred_cols = [
    'latitude', 'longitude', 'datetime', 'violation_type',
    'police_station', 'location_or_junction', 'cluster_id',
    'actual_high_risk', 'predicted_high_risk', 'predicted_probability_high_risk'
]
pred_cols = [c for c in pred_cols if c in df.columns]

df[pred_cols].to_csv(PREDICTIONS_CSV, index=False)
print(f'ML predictions saved: {PREDICTIONS_CSV}')
print(f'Shape: {df[pred_cols].shape}')
display(df[pred_cols].head())

ML predictions saved: ../data/processed/ml_predictions.csv
Shape: (298450, 10)


,latitude,longitude,datetime,violation_type,police_station,location_or_junction,cluster_id,actual_high_risk,predicted_high_risk,predicted_probability_high_risk
0,12.925557,77.618665,2023-11-20 00:28:46+00:00,"[""Wrong Parking"",""Parking Near Road Crossing""]",Madiwala,"18Th Main Road, Block 2, Koramangala, Bengalur...",0,0,0,0.0089
1,12.905463,77.700778,2023-11-24 22:46:46+00:00,"[""No Parking""]",Bellandur,"Sarjapura Main Road, The Grove, Janatha Colony...",1,0,0,0.0000
2,12.925449,77.618504,2023-11-20 00:27:46+00:00,"[""Wrong Parking"",""Parking In A Main Road""]",Madiwala,"Koramangala 2Nd Block, Kormangala West, Bengal...",0,0,0,0.0274
3,12.956521,77.518618,2023-11-16 06:47:46+00:00,"[""No Parking""]",Byatarayanapura,"6Th Cross Road, Manasa Layout, Nagarbhavi, Ben...",2,0,0,0.0887
4,12.977767,77.580545,2023-11-22 04:56:46+00:00,"[""No Parking""]",Upparpet,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",3,1,1,0.9906
